# Long-Term Trajectory Inference API - Example Usage

This notebook demonstrates how to use the Long-Term Trajectory Inference API
for predicting maritime trajectories using diffusion models.

The API:
- Downloads parquet files from S3
- Processes them (runs inference)
- Saves results back to S3 under `bucket/username/`



In [ ]:
import requests
import json
import time
from pathlib import Path

# Configure API base URL
# For local development: "http://localhost:8000"
# For production: Replace with your API URL
BASE_URL = "http://localhost:8000"
API_BASE = f"{BASE_URL}/api/v1"

print(f"[LOG] API Base URL: {API_BASE}")



## 1. Health Check

First, verify the API is running and accessible.



In [ ]:
url = f"{API_BASE}/health"

try:
    response = requests.get(url)
    response.raise_for_status()
    result = response.json()
    print("[LOG] API is healthy [DONE]")
    print(json.dumps(result, indent=2))
except requests.exceptions.RequestException as e:
    print(f"[LOG] Error connecting to API: {e}")
    print("Make sure the API is running: uvicorn app.main:app --reload")



## 2. Run Trajectory Inference

Predict future trajectories from a parquet file stored in S3.

**Requirements:**
- Input parquet file must exist in S3
- File should contain trajectory data with columns: `latitude`, `longitude`, and optionally `speed`, `course`, `heading`, `status`, `timestamp`
- Results will be saved to `s3://bucket/{username}/{output_filename}`



In [ ]:
url = f"{API_BASE}/predict"

# Example request payload
payload = {
    "s3_input_path": "username/raw_data/trajectory_data.parquet",  # Path to input file in S3
    "username": "john_doe",  # Username for output path
    "output_filename": "trajectory_inference_result.parquet"  # Optional: defaults to {input_name}_inference.parquet
}

print("[LOG] Sending prediction request...")
print(f"Input file: {payload['s3_input_path']}")
print(f"Output will be saved to: {payload['username']}/{payload['output_filename']}")
print()

try:
    response = requests.post(url, json=payload)
    response.raise_for_status()
    result = response.json()
    
    print("[LOG] Prediction completed successfully [DONE]")
    print(json.dumps(result, indent=2))
    
except requests.exceptions.HTTPError as e:
    print(f"[LOG] Error: {e}")
    if response.status_code == 400:
        print("Bad request - check your input parameters")
    elif response.status_code == 500:
        print("Server error - check API logs")
    try:
        print(response.json())
    except:
        print(response.text)
except requests.exceptions.RequestException as e:
    print(f"[LOG] Connection error: {e}")



## 3. Example: Process Multiple Files

Process multiple trajectory files in a batch.



In [ ]:
# List of files to process
files_to_process = [
    {
        "s3_input_path": "username/raw_data/trajectory_2024_01.parquet",
        "username": "john_doe",
        "output_filename": "trajectory_2024_01_inference.parquet"
    },
    {
        "s3_input_path": "username/raw_data/trajectory_2024_02.parquet",
        "username": "john_doe",
        "output_filename": "trajectory_2024_02_inference.parquet"
    },
]

url = f"{API_BASE}/predict"
results = []

for i, file_config in enumerate(files_to_process, 1):
    print(f"\n[LOG] Processing file {i}/{len(files_to_process)}: {file_config['s3_input_path']}")
    
    try:
        response = requests.post(url, json=file_config)
        response.raise_for_status()
        result = response.json()
        results.append(result)
        print(f"[LOG] ✓ Completed: {result['output_file']}")
    except requests.exceptions.HTTPError as e:
        print(f"[LOG] ✗ Failed: {e}")
        results.append({"error": str(e), "file": file_config['s3_input_path']})
    except Exception as e:
        print(f"[LOG] ✗ Unexpected error: {e}")
        results.append({"error": str(e), "file": file_config['s3_input_path']})

print(f"\n[LOG] Batch processing complete: {len([r for r in results if 'error' not in r])}/{len(results)} successful")



In [ ]:
def predict_trajectory(s3_input_path, username, output_filename=None, api_base=API_BASE):
    """
    Helper function to predict trajectory from S3 file.
    
    Args:
        s3_input_path: S3 path to input parquet file
        username: Username for output path
        output_filename: Optional output filename
        api_base: API base URL
        
    Returns:
        dict: API response
    """
    url = f"{api_base}/predict"
    payload = {
        "s3_input_path": s3_input_path,
        "username": username,
    }
    if output_filename:
        payload["output_filename"] = output_filename
    
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        return {"error": str(e), "status_code": response.status_code}
    except Exception as e:
        return {"error": str(e)}


# Example usage:
# result = predict_trajectory(
#     s3_input_path="username/raw_data/data.parquet",
#     username="john_doe",
#     output_filename="result.parquet"
# )
# print(result)



In [ ]:
def check_api_health(api_base=API_BASE):
    """Check if API is healthy."""
    url = f"{api_base}/health"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e), "status": "unhealthy"}


# Example usage:
# health = check_api_health()
# print(health)



## 5. Notes

- **Input files**: Must be parquet files stored in S3 with trajectory data
- **Output location**: Results saved to `s3://bucket/{username}/{output_filename}`
- **Processing**: Runs synchronously (waits for completion)
- **Error handling**: Check response status and error messages
- **API docs**: Visit `http://localhost:8000/docs` for interactive API documentation

**Required columns in input parquet:**
- `latitude` (required)
- `longitude` (required)
- `speed`, `course`, `heading`, `status`, `timestamp` (optional)

**Example input file structure:**
```
latitude | longitude | speed | course | heading | status | timestamp
---------|-----------|-------|--------|---------|--------|----------
35.0     | 135.0    | 10.5  | 45.0   | 45.0    | 0      | 1640995200
35.1     | 135.1    | 11.0  | 46.0   | 46.0    | 0      | 1640995500
...
```

